## Configure paths and initialize a Keypoint-MoSeq project from an existing project.

In [ ]:
import keypoint_moseq as kpms
import matplotlib.pyplot as plt
from pathlib import Path

WORKDIR = Path.cwd().parent
project_dir = WORKDIR / "results" / "B1R"

# dlc_project_dir = WORKDIR.parent / "dlc-pose-estimation" / "ElevatedMazeFood-Atanu-2026-04-04"
# dlc_config = dlc_project_dir / "config.yaml"

slp_project_dir = Path("/mnt/s/Projects/KY_Moseq/B1R")
sleap_file = slp_project_dir / "Mouse F1 Box 1 08-12-25 Reward Training Day 1.predictions.analysis.h5"

print(f"slp_project_dir: {slp_project_dir}")
print(f"kpms project_dir: {project_dir}")

In [ ]:
# Create the KPMS project once, then define a relaxed config loader.
# if not (project_dir / "config.yml").exists():
#     kpms.setup_project(project_dir, deeplabcut_config=str(dlc_config), overwrite=True)

# # Use a relaxed load first in case anatomy fields still need editing.
# config = lambda: kpms.load_config(project_dir)

if not (project_dir / "config.yml").exists():
    kpms.setup_project(str(project_dir), sleap_file=str(sleap_file))

# Use a relaxed load first in case anatomy fields still need editing.
config = lambda: kpms.load_config(project_dir)

In [ ]:
# Set core project config values for this dataset.
kpms.update_config(
    project_dir,
    # video_dir=str(dlc_project_dir / "videos"),
    video_dir=str(slp_project_dir),
    anterior_bodyparts=["nose"],
    posterior_bodyparts=["TailBase"],
    fps=30,
)

# After anatomy values are valid, switch back to strict config loading.
config = lambda: kpms.load_config(project_dir)

## Load Data
Load raw keypoints and keep untouched backups so selected keypoints can be restored later if needed.

In [ ]:
# # load data (e.g. from DeepLabCut)
# keypoint_data_path = str(dlc_project_dir / "raw_pose_data")  # can be a file, a directory, or a list of files
# coordinates, confidences, bodyparts = kpms.load_keypoints(keypoint_data_path, "deeplabcut")

# load data (e.g. from SLEAP)
# keypoint_data_path = str(slp_project_dir)  # can be a file, a directory, or a list of files
keypoint_data_path = [str(item) for item in slp_project_dir.glob("*.h5") if item.is_file()][:5]  # limit to first 10 files
coordinates, confidences, bodyparts = kpms.load_keypoints(keypoint_data_path, "sleap", extension=".h5")

# Preserve originals for optional per-keypoint restoration after outlier interpolation.
orig_coordinates = {k: v.copy() for k, v in coordinates.items()}
orig_confidences = {k: v.copy() for k, v in confidences.items()}

## Remove outlier keypoints
Run medoid-distance outlier interpolation, then restore original Midback values to avoid over-correction of a central keypoint.

In [ ]:
# Configure outlier sensitivity and run outlier interpolation on temporary copies.
kpms.update_config(project_dir, outlier_scale_factor=6.0)

# outlier_removal mutates inputs in place, so work on temporary copies.
temp_coordinates = {k: v.copy() for k, v in coordinates.items()}
temp_confidences = {k: v.copy() for k, v in confidences.items()}

temp_coordinates, temp_confidences = kpms.outlier_removal(
    temp_coordinates,
    temp_confidences,
    project_dir,
    overwrite=True,
    **config()
)

# Bodyparts you want to restore from originals
restore_parts = ["bodym", "bodyL", "bodyR"]  # edit this list
restore_idx = [bodyparts.index(bp) for bp in restore_parts]

for rec in temp_coordinates:
    for idx in restore_idx:
        temp_coordinates[rec][:, idx, :] = orig_coordinates[rec][:, idx, :]
        temp_confidences[rec][:, idx] = orig_confidences[rec][:, idx]

coordinates, confidences = temp_coordinates, temp_confidences

## Save cleaned keypoints so later steps can resume without rerunning preprocessing.

In [ ]:
import pickle

cleaned_snapshot = project_dir / "cleaned_keypoints.pkl"
with open(cleaned_snapshot, "wb") as f:
    pickle.dump(
        {
            "coordinates": coordinates,
            "confidences": confidences,
            "bodyparts": bodyparts,
        },
        f,
    )

print(f"Saved cleaned keypoints snapshot: {cleaned_snapshot}")

## Format data for modeling
Optionally reload the cleaned snapshot, then convert keypoints into model-ready arrays.

In [ ]:
# Optional resume path: use this after kernel restart to skip preprocessing cells.
import pickle

cleaned_snapshot = project_dir / "cleaned_keypoints.pkl"
if cleaned_snapshot.exists():
    with open(cleaned_snapshot, "rb") as f:
        snap = pickle.load(f)
    coordinates = snap["coordinates"]
    confidences = snap["confidences"]
    bodyparts = snap["bodyparts"]
    print(f"Loaded cleaned keypoints snapshot: {cleaned_snapshot}")
else:
    print(f"Snapshot not found: {cleaned_snapshot}")

In [ ]:
# Build batched arrays and metadata used by downstream KPMS model fitting.
data, metadata = kpms.format_data(coordinates, confidences, **config())

In [ ]:
import pickle

formatted_snapshot = project_dir / "formatted_data.pkl"
with open(formatted_snapshot, "wb") as f:
    pickle.dump({"data": data, "metadata": metadata}, f)

print(f"Saved formatted snapshot: {formatted_snapshot}")

## Update sigmasq_loc

In [ ]:
estimated_sigmasq_loc = kpms.estimate_sigmasq_loc(
    data["Y"], data["mask"], filter_size=config()["fps"]
)
print("estimated sigmasq_loc:", estimated_sigmasq_loc)

kpms.update_config(project_dir, sigmasq_loc=float(estimated_sigmasq_loc))
print("updated sigmasq_loc:", config()["cen_hypparams"]["sigmasq_loc"])

## Calibration
Estimate confidence-to-error mapping parameters using manual annotation support.

In [ ]:
%matplotlib widget
kpms.noise_calibration(project_dir, coordinates, confidences, **config())

## Fit PCA

In [ ]:
plt.close("all")
%matplotlib inline
pca = kpms.fit_pca(**data, **config())
kpms.save_pca(pca, project_dir)

kpms.print_dims_to_explain_variance(pca, 0.9)
kpms.plot_scree(pca, project_dir=project_dir)
kpms.plot_pcs(pca, project_dir=project_dir, **config())

# use the following to load an already fit model
# pca = kpms.load_pca(project_dir)

## Set latent dimensionality before model initialization

In [ ]:
kpms.update_config(project_dir, latent_dim=2)